# Access the rem silver_data

In [0]:
df_silver=spark.sql("select * from parquet.`abfss://silver@storagecarsud1.dfs.core.windows.net/cars_data_silver`")
       

# Access the dim data

In [0]:
df_model=spark.sql("select * from cars_catalog.gold.dim_model")
df_dealer=spark.sql("select * from cars_catalog.gold.dim_dealer")
df_branch=spark.sql("select * from cars_catalog.gold.dim_branch")
df_date=spark.sql("select * from cars_catalog.gold.dim_date")


# join the data_frames

In [0]:
df_fact=df_silver.join(df_branch,df_branch['Branch_ID']==df_silver['Branch_ID'],how='inner')\
    .join(df_dealer,df_dealer['Dealer_ID']==df_silver['Dealer_ID'],'left')\
        .join(df_model,df_model['Model_Id']==df_silver['Model_ID'],'left')\
            .join(df_date,df_date['Date_ID']==df_silver['Date_ID'],'left')\
                .select(df_silver['Revenue'],df_silver['Units_Sold'],df_silver['revenue_perUnit'],df_branch['dim_branch_key'],df_dealer['dim_dealer_key'],df_model['dim_model_key'],df_date['dim_date_key'])

# Save the data

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("cars_catalog.gold.fact_table"):
    delta_tbl=DeltaTable.forPath(spark,path="abfss://gold@storagecarsud1.dfs.core.windows.net/fact_table")
    delta_tbl.alias('t').merge(df_fact.alias('src'),'t.dim_date_key=src.dim_date_key and t.dim_branch_key=src.dim_branch_key and t.dim_dealer_key=src.dim_dealer_key and t.dim_model_key=src.dim_model_key')\
        .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df_fact.write.format('delta')\
        .mode('overwrite')\
            .option('path','abfss://gold@storagecarsud1.dfs.core.windows.net/fact_table')\
                .saveAsTable('cars_catalog.gold.fact_table')